<a href="https://colab.research.google.com/github/cernestovelezo230161-hue/Carlos_Velez_O/blob/main/EDA_Assignment_Taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA Notebook Assignment: NYC Taxi Rides (25% of the final grade)
**Individual, take-home.** You get one messy file, `taxi_rides_messy.csv`: about 6,400 taxi rides in New York City, picked up between 28 February and 31 March 2019. Your job is to **clean it, explore it and explain what you find**.

**One row = one taxi ride.**

| Column | Meaning |
|---|---|
| `ride_id` | ride identifier |
| `pickup`, `dropoff` | date and time the ride started and ended |
| `passengers` | number of passengers (entered by the driver) |
| `distance` | trip distance in miles |
| `fare` | meter fare in dollars (without tip, tolls and taxes) |
| `tip`, `tolls`, `total` | tip, tolls and total amount paid, in dollars |
| `color` | `yellow` taxi or `green` taxi (green taxis serve mostly outside central Manhattan) |
| `payment` | `credit card` or `cash` |
| `pickup_zone`, `dropoff_zone` | neighbourhood |
| `pickup_borough`, `dropoff_borough` | borough (Manhattan, Queens, Brooklyn, Bronx; 2 dropoffs in Staten Island) |

**How to work**
- Run the cells in order (Shift + Enter). In Colab: Runtime > Run all.
- Write your code in the empty cells. Run the **test cell** under each code exercise: every line should print `True`. Passing tests earns the code points; plots and written answers are graded by the instructor.
- Answer the questions marked **Your answer** in the markdown cell under them (double-click to edit). Use numbers from your output.
- Do **not** rename the variables we ask for: the tests use them.
- Points are shown in each heading. Total: **100 points** = 25% of the final grade.

**Submit:** your `.ipynb` file, run from top to bottom (Runtime > Restart and run all) with all outputs visible. Name it `EDA_Lastname_Firstname.ipynb`.

## Setup
**In Colab:** click the folder icon on the left, then the upload icon, and upload `taxi_rides_messy.csv`. (In Jupyter, put the file in the same folder as this notebook.)

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('https://raw.githubusercontent.com/cernestovelezo230161-hue/Carlos_Velez_O/main/taxi_rides_messy.csv')

print(df.shape)
df.head()

(6468, 15)


,ride_id,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,R04190,2019-03-01 18:29:07,2019-03-01 19:27:15,1,16.96,52.0,13.11,5.76,78.67,yellow,credit card,Union Sq,JFK Airport,Manhattan,Queens
1,R05846,2019-03-06 10:52:11,2019-03-06 10:54:59,1,0.52,4.0,0.00,0.00,4.80,green,cash,Central Harlem,Central Harlem,Manhattan,Manhattan
2,R03773,2019-03-08 14:04:17,2019-03-08 14:21:24,2,1.47,10.5,0.00,0.00,13.80,yellow,cash,Penn Station/Madison Sq West,Kips Bay,Manhattan,Manhattan
3,R02382,2019-03-10 14:20:21,2019-03-10 14:26:17,1,0.8,6.0,1.85,0.00,11.15,yellow,credit card,West Village,East Chelsea,Manhattan,Manhattan
4,R03678,2019-03-17 09:43:30,2019-03-17 09:54:45,1,1.5,9.5,2.00,0.00,14.80,yellow,credit card,Gramercy,West Chelsea/Hudson Yards,Manhattan,Manhattan


In [ ]:




































































































































from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
# Part 1: First look (5 points)

### 1.1 Shape, types and problems (3 points)
1. `n_rows`, `n_cols`: the number of rows and columns of `df`.
2. `n_dup`: the number of rows that are **exact duplicates** of an earlier row.
3. `missing`: a Series with the number of missing values in each column (`isna().sum()`).

Then run `df.info()` and look at the type of each column.

In [9]:
n_rows, n_cols = None, None
n_dup = None
missing = None


n_rows, n_cols = df.shape
print("Rows=", n_rows)
print("Columns=", n_cols)

n_dup = df.duplicated().sum()
print("n_dup=", n_dup)

missing = df.isna().sum()
print("missing=", missing)

Rows: 6468
Columns: 15
Duplicates: 35

Missing values per column:
 ride_id             0
pickup              0
dropoff             0
passengers          0
distance            0
fare                0
tip                 0
tolls               0
total               0
color               0
payment            44
pickup_zone        26
dropoff_zone       46
pickup_borough     26
dropoff_borough    46
dtype: int64


In [10]:
# Test cell: run me
print(n_rows == 6468 and n_cols == 15)
print(n_dup == 35)
print(missing['payment'] == 44 and missing['fare'] == 0)

True
True
True


### 1.2 Question (2 points)
`distance` should be a number, but `df.info()` does not show it as a numeric type (`float64`). Why? Find the reason with `df['distance'].value_counts().tail(15)` (the rarest values), and describe it in one or two sentences.

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6468 entries, 0 to 6467
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ride_id          6468 non-null   object 
 1   pickup           6468 non-null   object 
 2   dropoff          6468 non-null   object 
 3   passengers       6468 non-null   int64  
 4   distance         6468 non-null   object 
 5   fare             6468 non-null   float64
 6   tip              6468 non-null   float64
 7   tolls            6468 non-null   float64
 8   total            6468 non-null   float64
 9   color            6468 non-null   object 
 10  payment          6424 non-null   object 
 11  pickup_zone      6442 non-null   object 
 12  dropoff_zone     6422 non-null   object 
 13  pickup_borough   6442 non-null   object 
 14  dropoff_borough  6422 non-null   object 
dtypes: float64(4), int64(1), object(10)
memory usage: 758.1+ KB


In [12]:
non_num_rows = df[pd.to_numeric(df['distance'], errors='coerce').isna() & df['distance'].notna()]

print(non_num_rows['distance'].unique())

['unknown']


**Your answer:** The reason the column distance does not show to be a numeric type is because at least one value in the dataset is not a numeric value. After runing the code to find the rows with non numeric values, it was found "unknown" which is a string

In [ ]:
df['distance'].value_counts().tail(15)

The rarest value of distance found in the last 15 rows is 0.18, which is rather a short distance to take a taxi instead of walking.

---
# Part 2: Cleaning (25 points)
Clean a **copy** called `rides`. Follow the rules exactly so the tests can check your work. Keep `df` unchanged.

### 2.1 Duplicates (2 points)
Create `rides`: `df` without exact duplicate rows. Use `.copy()` at the end.

In [ ]:

rides = df.drop_duplicates().copy()

In [ ]:
# Test cell: run me
print(len(rides) == 6433)
print(rides['ride_id'].duplicated().sum() == 0)

### 2.2 Inconsistent text (3 points)
Look at `rides['color'].value_counts()` and `rides['payment'].value_counts()`: the same category is written in several ways.
Clean both columns with `.str.strip()` and `.str.lower()` so that `color` only contains `'yellow'` and `'green'`, and `payment` only `'credit card'` and `'cash'` (plus missing values).

In [ ]:
# your code here
rides['color'].value_counts()

In [ ]:
rides['payment'].value_counts()

In [ ]:
rides['color'] = rides['color'].str.strip().str.lower()
rides['payment'] = rides['payment'].str.strip().str.lower()
print(rides['color'].value_counts(dropna=False))

In [ ]:
print(rides['payment'].value_counts(dropna=False))

In [ ]:
# Test cell: run me
print(sorted(rides['color'].unique()) == ['green', 'yellow'])
print(sorted(rides['payment'].dropna().unique()) == ['cash', 'credit card'])
print((rides['color'] == 'yellow').sum() == 5451)

### 2.3 Text in a numeric column (3 points)
Convert `distance` to numbers with `pd.to_numeric(..., errors='coerce')`. Values that are not numbers become `NaN`.
`n_bad_distance`: how many values became `NaN`.

In [ ]:
#import pandas as pd

# 1. Keep track of values that were NOT already missing before conversion
was_present = df['distance'].notna()

# 2. Convert distance to numbers, turning non-numeric values into NaN
df['distance'] = pd.to_numeric(df['distance'], errors='coerce')

# 3. Count how many values became NaN during conversion
n_bad_distance = (was_present & df['distance'].isna()).sum()

#n_bad_distance = None

In [ ]:
# Test cell: run me
print(pd.api.types.is_numeric_dtype(rides['distance']))
print(n_bad_distance == 12)

### 2.4 Impossible values (4 points)
Set these values to `NaN` with `rides.loc[condition, column] = np.nan`:
- `fare` below 0 (a fare cannot be negative)
- `distance` above 100 miles (a typing error: no ride in this city is that long)
- `passengers` equal to 0 (the driver did not enter the number)

Before you replace them, store the **number** of rows found in `n_neg_fare`, `n_long` and `n_zero_pass`.

In [ ]:
n_neg_fare = None
n_long = None
n_zero_pass = None
# replace with NaN here

In [ ]:
# Test cell: run me
print(n_neg_fare == 9 and n_long == 5 and n_zero_pass == 96)
print(rides['fare'].min() >= 0 and rides['distance'].max() <= 100)
print((rides['passengers'] == 0).sum() == 0)

### 2.5 Dates and new time columns (5 points)
1. Convert `pickup` and `dropoff` to dates with `pd.to_datetime`.
2. Add `duration_min`: the ride length in minutes.
   *Hint:* `(rides['dropoff'] - rides['pickup']).dt.total_seconds() / 60`
3. Some rides end **before** they start (the two times were typed in the wrong order). **First** store in `n_neg_duration` how many values of `duration_min` are below 0, **then** set them to `NaN`.
4. Add `hour`: the hour of the pickup (0 to 23), with `.dt.hour`.
5. Add `day`: the name of the pickup day, with `.dt.day_name()`.

In [ ]:
# your code here
n_neg_duration = None

In [ ]:
# Test cell: run me
print(str(rides['pickup'].dtype).startswith('datetime64'))
print(n_neg_duration == 7 and rides['duration_min'].min() >= 0)
print(rides['hour'].max() == 23 and 'Friday' in set(rides['day']))

### 2.6 Missing values (3 points)
1. Fill missing `payment` with the text `'unknown'` (we do not know how these rides were paid, so we do not guess).
2. Drop the rows where `distance` **or** `fare` is missing (`dropna(subset=[...])`, then `.copy()`).
3. Keep the other missing values (zones, boroughs, `passengers`, `duration_min`): the rest of each row is still useful.

In [ ]:
# your code here

In [ ]:
# Test cell: run me
print(len(rides) == 6407)
print(rides['payment'].isna().sum() == 0 and (rides['payment'] == 'unknown').sum() == 44)
print(rides['distance'].isna().sum() == 0 and rides['fare'].isna().sum() == 0)

### 2.7 A fair comparison column (2 points)
A $5 tip on a $10 fare is generous, on a $60 fare it is not. Add `tip_pct`: the tip as a percentage of the fare (`tip / fare * 100`). Keep it for all rides for now; in Part 3 you will see why it is only meaningful for some of them.

In [ ]:
# your code here

In [ ]:
# Test cell: run me
print('tip_pct' in rides.columns)
print(round(rides['tip_pct'].median(), 1) == 21.1)

### 2.8 Cleaning log (3 points)
Complete the table: one line per problem you fixed. "Rows" is how many rows were affected.

| Problem | How I found it | What I did | Rows |
|---|---|---|---|
| ... | ... | ... | ... |

| Problem | How I found it | What I did | Rows |
|---|---|---|---|
| ... | ... | ... | ... |

**Check your cleaning.** Run the cell below: it must run without an error. Every part from here uses `rides`.

In [ ]:
assert rides['distance'].between(0, 100).all()
assert (rides['fare'] >= 0).all()
assert rides['color'].isin(['yellow', 'green']).all()
assert rides['payment'].isin(['credit card', 'cash', 'unknown']).all()
print('Clean data:', rides.shape)

---
# Part 3: Descriptive statistics (15 points)

### 3.1 Numeric summary (3 points)
`summary`: `describe()` of the columns `fare`, `distance`, `duration_min` and `tip`, rounded to 2 decimals.

In [ ]:
summary = None
summary

In [ ]:
# Test cell: run me
print(summary.loc['50%', 'fare'] == 9.5 and summary.loc['mean', 'fare'] == 13.09)
print(summary.loc['count', 'duration_min'] == 6400)

**Question (2 points).** For `fare`, compare the mean and the median. Which one better describes a "typical" ride, and why? What does the gap tell you about the shape of the distribution?

**Your answer:** ...

### 3.2 Categories (3 points)
1. `pay_share`: the **percentage** of rides for each `payment` value, rounded to 1 decimal (`value_counts(normalize=True)`).
2. `borough_counts`: the number of rides per `pickup_borough`.

In [ ]:
pay_share = None
borough_counts = None

In [ ]:
# Test cell: run me
print(pay_share['credit card'] == 71.2)
print(borough_counts.idxmax() == 'Manhattan' and borough_counts['Manhattan'] == 5246)

### 3.3 Comparing groups (3 points)
`by_color`: for each `color`, the number of rides `n`, the mean distance `mean_distance` and the median fare `median_fare`, rounded to 2 decimals. Use named aggregation:
`rides.groupby('color').agg(n=('fare', 'count'), ...)`

In [ ]:
by_color = None
by_color

In [ ]:
# Test cell: run me
print(list(by_color.columns) == ['n', 'mean_distance', 'median_fare'])
print(by_color.loc['green', 'n'] == 980 and by_color.loc['green', 'mean_distance'] == 3.41)

### 3.4 Tips by payment type (2 points)
`tips_by_pay`: for each `payment` value, the mean tip `mean_tip` and the share of rides with a tip of exactly 0, `zero_tip_share`, rounded to 2 decimals.
*Hint:* the share of zeros is the mean of `True`/`False` values: `('tip', lambda s: (s == 0).mean())`.

In [ ]:
tips_by_pay = None
tips_by_pay

In [ ]:
# Test cell: run me
print(tips_by_pay.loc['cash', 'zero_tip_share'] == 1.0)
print(tips_by_pay.loc['credit card', 'mean_tip'] == 2.78)

**Question (2 points).** Every cash ride has a tip of 0. Do cash customers really never tip? Give the most likely explanation, and say what it means for any analysis of tips.

**Your answer:** ...

---
# Part 4: Distributions (15 points)
Every plot needs a **title** and **axis labels with units**.

### 4.1 Histogram and skewness (4 points)
1. Draw a histogram of `fare` with 50 bins.
2. `skews`: the skewness of `fare`, `distance`, `duration_min` and `tip`, rounded to 2 decimals.
3. `most_skewed`: the name of the column with the largest skewness (`idxmax()`).

In [ ]:
# your plot here
skews = None
most_skewed = None

In [ ]:
# Test cell: run me
print(skews['fare'] == 3.22)
print(most_skewed == 'fare')

**Question (2 points).** Describe the shape of the fare distribution in one or two sentences (where most values are, the tail, any unusual peaks). Look closely: is there a spike far from the rest? What could it be?

**Your answer:** ...

### 4.2 Box plots by group (4 points)
1. Draw box plots of `fare` for each pickup borough, in the order `['Manhattan', 'Brooklyn', 'Queens', 'Bronx']`, with the borough names under the boxes.
2. `fare_medians`: the median fare per `pickup_borough` (groupby).
3. `top_borough`: the borough with the highest median fare.

In [ ]:
boroughs = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx']
# your plot here
fare_medians = None
top_borough = None

In [ ]:
# Test cell: run me
print(fare_medians['Manhattan'] == 8.5)
print(top_borough == 'Queens')

### 4.3 When do people ride? (3 points)
1. `by_hour`: the number of rides per `hour`, sorted by hour (`value_counts().sort_index()`).
2. Draw it as a **bar chart** (`plt.bar(by_hour.index, by_hour.values)`) with a title and labels.
3. `busiest_hour`: the hour with the most rides.

In [ ]:
by_hour = None
# your plot here
busiest_hour = None

In [ ]:
# Test cell: run me
print(len(by_hour) == 24 and by_hour.sum() == 6407)
print(busiest_hour == 18)

**Question (2 points).** Describe the daily pattern in your bar chart. When is it quiet, when is it busy?

**Your answer:** ...

---
# Part 5: Correlation (15 points)

### 5.1 Correlation matrix and heatmap (4 points)
1. `corr`: the correlation matrix of `passengers`, `distance`, `duration_min`, `fare`, `tip`, `tolls` and `total`, rounded to 2 decimals.
2. Draw it as a heatmap with `plt.imshow` (`cmap='coolwarm'`, `vmin=-1`, `vmax=1`), a colorbar and the column names on both axes.
3. `r_dist_fare`: the correlation between `distance` and `fare`, read from `corr`.

In [ ]:
cols = ['passengers', 'distance', 'duration_min', 'fare', 'tip', 'tolls', 'total']
corr = None
# your heatmap here
r_dist_fare = None

In [ ]:
# Test cell: run me
print(corr.shape == (7, 7))
print(r_dist_fare == 0.92)

**Question (3 points).**
1. Which variable is almost unrelated to all the others? Why does that make sense?
2. `fare` and `total` have r = 0.97. Is that an interesting discovery? Explain.

**Your answer:** ...

### 5.2 Pearson or Spearman? (2 points)
Compute the correlation between `distance` and `duration_min` with both methods, rounded to 2 decimals: `r_pearson` and `r_spearman` (use `method='spearman'`). Missing values are skipped automatically.

In [ ]:
r_pearson = None
r_spearman = None

In [ ]:
# Test cell: run me
print(r_pearson == 0.82)
print(r_spearman == 0.87)

### 5.3 Check the correlation inside a group (3 points)
1. `r_all`: the correlation between `fare` and `tip` for **all** rides.
2. `r_card`: the same correlation for **credit card rides only**.
Round both to 2 decimals.

In [ ]:
r_all = None
r_card = None

In [ ]:
# Test cell: run me
print(r_all == 0.49)
print(r_card == 0.61)

**Question (3 points).** Why is `r_card` higher than `r_all`? Use your answer to 3.4. Which number should you report, and what is the general lesson?

**Your answer:** ...

---
# Part 6: Visualization (15 points)

### 6.1 Scatter plot by group (4 points)
Draw a scatter plot of `fare` (y) against `distance` (x), with yellow taxis and green taxis in two colours (one `plt.scatter` per colour, with `label=` and `plt.legend()`). Use small, transparent dots (`s=8`, `alpha=0.4`). Add a title that states the main point.
Then set `pattern` to the best description: `'linear'`, `'curved'` or `'none'`.

In [ ]:
# your plot here
pattern = None

In [ ]:
# Test cell: run me
print(pattern == 'linear')

### 6.2 Two categories together (4 points)
1. `color_mix`: `pd.crosstab(rides['pickup_borough'], rides['color'], normalize='index')`, multiplied by 100 and rounded to 1 decimal: the percentage of yellow and green taxis in each borough.
2. Draw it as a bar chart: `color_mix.plot(kind='bar', color=['seagreen', 'gold'])`, with a title and a y label.

In [ ]:
color_mix = None
# your plot here

In [ ]:
# Test cell: run me
print(color_mix.loc['Manhattan', 'yellow'] == 94.4)
print(color_mix.loc['Bronx', 'green'] == 83.8)

### 6.3 A one-figure summary (7 points)
Make **one** figure with **4 plots** in a 2 x 2 grid: `fig, axes = plt.subplots(2, 2, figsize=(11, 8))`. Use `axes[0, 0]`, `axes[0, 1]`, ... and `.set_title`, `.set_xlabel`, `.set_ylabel`.
Choose four plots that together tell the main story of this data (you may reuse ideas from above or make new ones, for example rides per day of the week, or tip % by borough for card rides). Add an overall title with `fig.suptitle(...)` and call `plt.tight_layout()`.

Graded on: 4 different, relevant plots (3), titles and labelled axes with units (2), clean and readable (2).
`fig` must be your figure.

In [ ]:
fig = None
# your figure here

In [ ]:
# Test cell: run me
print(fig is not None and len(fig.axes) >= 4)
print(all(ax.get_title() != '' for ax in fig.axes[:4]))

---
# Part 7: Findings (10 points)

### 7.1 Your findings (8 points)
Write **four** findings in the list `findings`. Each one must:
- be a full sentence with at least **one number** from your analysis,
- say **what it means** (the "so what"),
- point to the plot or table that supports it (for example "see 4.3").

At least one finding must be about a **relationship** between two variables.

In [ ]:
findings = [
    '...',
    '...',
    '...',
    '...',
]

In [ ]:
# Test cell: run me
print(len(findings) == 4)
print(all(any(ch.isdigit() for ch in f) for f in findings))
print(all(f != '...' for f in findings))

### 7.2 One limitation (2 points)
Name **one** limitation of this data or of your analysis, and say how it could change a conclusion.

**Your answer:** ...

---
### Before you submit
| Check | ✔ |
|---|---|
| Runtime > Restart and run all: no errors | |
| Every test cell prints only `True` | |
| Every plot has a title and axis labels with units | |
| Every **Your answer** cell is completed with numbers | |
| File named `EDA_Lastname_Firstname.ipynb` | |